<a href="https://colab.research.google.com/github/Llamachan/youtube-spam-enforcement/blob/main/01_load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
client = bigquery.Client(project="yt-spam-study")
print(client.project)

yt-spam-study


In [25]:
!git clone https://github.com/Llamachan/youtube-spam-enforcement.git

fatal: destination path 'youtube-spam-enforcement' already exists and is not an empty directory.


In [26]:
import glob, os
import pandas as pd

frames = []
for path in sorted(glob.glob("youtube-spam-enforcement/data/raw/Youtube0*.csv")):
    df = pd.read_csv(path)
    # Youtube01-Psy.csv -> Psy
    df["video"] = os.path.basename(path).split("-")[1].removesuffix(".csv")
    frames.append(df)

comments = pd.concat(frames, ignore_index=True)
comments = comments.rename(columns={"CLASS": "is_spam"})
comments.columns = comments.columns.str.lower()

print(comments.shape)
comments.groupby(["video", "is_spam"]).size()

(1956, 6)


video      is_spam
Eminem     0          203
           1          245
KatyPerry  0          175
           1          175
LMFAO      0          202
           1          236
Psy        0          175
           1          175
Shakira    0          196
           1          174
dtype: int64

In [27]:
comments["date"] = pd.to_datetime(comments["date"], format="ISO8601", errors="coerce", utc=True)
print("Missing or bad dates:", comments["date"].isna().sum())

Missing or bad dates: 245


In [28]:
before = len(comments)
print("Duplicate comment_ids:", comments["comment_id"].duplicated().sum())
comments = comments.drop_duplicates("comment_id", keep="first")
print("Rows after removing duplicates:", len(comments))

Duplicate comment_ids: 3
Rows after removing duplicates: 1953


In [29]:
table_id = f"{client.project}.yt_spam.comments"
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(comments, table_id, job_config=job_config)
job.result()
print("Rows in BigQuery:", client.get_table(table_id).num_rows)

Rows in BigQuery: 1953


In [30]:
print(f"Rows loaded: {before}")
print(f"Duplicate comment_ids removed: {before - len(comments)}")
print(f"Missing or bad dates: {comments['date'].isna().sum()}")
print(f"Final rows: {len(comments)}")
print(f"Spam comments: {comments['is_spam'].sum()}")

Rows loaded: 1956
Duplicate comment_ids removed: 3
Missing or bad dates: 243
Final rows: 1953
Spam comments: 1003


Data Checks
Rows loaded: 1956
Duplicate comment_ids removed: 3
Missing or bad dates: 243
Final rows: 1953
Spam comments: 1003